# 09 · Binary task (benign vs attack): results for all three datasets

Run after notebooks 01 (or 08) and 02 for the binary profiles, e.g. `python run_all.py --binary --steps 8 2`. Needs no raw data.

Same CSV files and the same leakage-safe feature policy, but the target is the 0/1 `Label` column (0 = benign, 1 = attack, the positive class).
The multi-class `Attack` column is not used for training; it is used only at evaluation time to report the detection rate per attack type.

Binary and multi-class results are different tasks and are reported separately.

Outputs: `paper/figures/bin_*.png|pdf`, `paper/tables/bin_*.csv|tex`.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "paper_utils.py").exists():
    if ROOT.parent == ROOT: raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from paper_utils import *
set_paper_style()
RES, FIG, TAB = ROOT / "results", ROOT / "paper" / "figures", ROOT / "paper" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
BIN = [d + "_binary" for d in DATASETS]
present = [b for b in BIN if (RES / "article_results" / b / "article_results_summary.json").exists()]
lab = lambda b: DS_LABEL[b.replace("_binary", "")] + " (binary)"
col = lambda b: DS_COLOR[b.replace("_binary", "")]
print("binary results found for:", [lab(b) for b in present] or "none yet -> run notebooks 01 and 02 with the *_binary profiles")
assert present, "no binary results yet"
S  = {b: load_json(RES / "article_results" / b / "article_results_summary.json") for b in present}
PC = {b: read_csv_if(RES / "publishability_checks" / b / "per_class_metrics_saved_model.csv").set_index("class_name") for b in present}
CM = {b: read_csv_if(RES / "publishability_checks" / b / "confusion_matrix_saved_model.csv", index_col=0) for b in present}
BL = {b: read_csv_if(RES / "publishability_checks" / b / "baseline_comparison_exact_split.csv") for b in present}
AT = {b: read_csv_if(RES / "publishability_checks" / b / "attack_type_detection.csv") for b in present}
MS = {b: read_csv_if(RES / "article_results" / b / "multi_seed_repeats_summary.csv") for b in present}

## Table B1: binary results on the held-out test split

In [ ]:
rows = []
for b in present:
    m = S[b]["held_out_test_metrics"]; pc = PC[b]; cm = CM[b].values; tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    k = int(S[b]["validation_selected_k"]); nodes = int(m["node_count"])
    ms = MS[b]; mcc_ms = ""
    if ms is not None:
        r = ms[ms.iloc[:, 0] == "mcc"]
        if len(r): mcc_ms = f"{float(r['mean'].iloc[0]):.4f} ± {float(r['std'].iloc[0]):.4f}"
    rows.append({"Dataset": lab(b), "Test rows": int(cm.sum()), "Malicious %": 100 * (tp + fn) / cm.sum(), "Accuracy": m["accuracy"], "MCC": m["mcc"], "Kappa": m["kappa"],
                 "Precision (malicious)": pc.loc["Malicious", "precision"], "Recall (malicious)": pc.loc["Malicious", "recall"], "F1 (malicious)": pc.loc["Malicious", "f1"],
                 "Specificity (benign recall)": pc.loc["Benign", "recall"], "False-positive rate": fp / (fp + tn), "False-negative rate": fn / (fn + tp),
                 "Balanced accuracy": (pc.loc["Malicious", "recall"] + pc.loc["Benign", "recall"]) / 2, "K": k, "Nodes": nodes, "Depth": int(m["depth"]),
                 "Exported KB (est.)": (11 * nodes + 13 * k) / 1024, "MCC mean±std (seeds)": mcc_ms})
b1 = pd.DataFrame(rows); display(b1.T)
save_table(b1, TAB / "bin_tab01_headline", "Binary (benign vs malicious) held-out test results of the optimized decision tree.", "tab:bin_main", "{:.4f}")

## Table B2: selected tree vs baselines (same split)

In [ ]:
keep = ["Optimized_DT_HSDO_saved", "DT_default_all_features", "DT_default_best_k", "DT_default_all_features_FULLTRAIN", "DT_default_best_k_FULLTRAIN",
        "RandomForest_100_best_k", "ExtraTrees_100_best_k", "XGBoost_200_best_k", "KNN_5_best_k", "LogReg_balanced_best_k"]
fr = []
for b in present:
    t = BL[b].copy(); t.insert(0, "Dataset", lab(b)); fr.append(t[t.model.isin(keep)][["Dataset", "model", "accuracy", "macro_f1", "mcc", "node_count", "size_proxy_kb"]])
b2 = pd.concat(fr); b2.columns = ["Dataset", "Model", "Accuracy", "Macro-F1", "MCC", "Nodes", "Flash est. (KB)"]; display(b2)
save_table(b2, TAB / "bin_tab02_baselines", "Binary task: optimized tree vs same-split baselines. Rows named *_FULLTRAIN use all training rows; the others use a capped training subset.", "tab:bin_base", "{:.4f}")

## Table B3: detection rate per attack type

In [ ]:
fr = []
for b in present:
    if AT[b] is not None:
        t = AT[b].copy(); t.insert(0, "Dataset", lab(b)); fr.append(t)
if fr:
    b3 = pd.concat(fr); b3 = b3[["Dataset", "attack_type", "kind", "n_test", "flagged_malicious", "rate", "rate_meaning"]]
    b3.columns = ["Dataset", "Attack type", "Kind", "Test rows", "Flagged malicious", "Rate", "Meaning"]
    display(b3[b3.Kind == "attack"].sort_values(["Dataset", "Rate"]).head(30)); save_table(b3, TAB / "bin_tab03_attack_type_detection", "Detection rate per attack type (binary model); benign row = false-positive rate.", "tab:bin_atk", "{:.4f}")
else:
    print("no attack_type_detection.csv yet - re-run notebook 02 for the binary profiles")

## Table B4: binary vs multi-class on the same data (different tasks, for orientation)

In [ ]:
rows = []
for b in present:
    base = b.replace("_binary", ""); f = RES / "article_results" / base / "article_results_summary.json"
    if f.exists():
        mb, mm = S[b]["held_out_test_metrics"], load_json(f)["held_out_test_metrics"]
        rows.append({"Dataset": DS_LABEL[base], "Binary MCC": mb["mcc"], "Multi-class MCC": mm["mcc"], "Binary accuracy": mb["accuracy"], "Multi-class accuracy": mm["accuracy"],
                     "Binary nodes": int(mb["node_count"]), "Multi-class nodes": int(mm["node_count"])})
if rows:
    b4 = pd.DataFrame(rows); display(b4); save_table(b4, TAB / "bin_tab04_binary_vs_multiclass", "Binary vs multi-class results (different tasks; not directly comparable).", "tab:bin_vs", "{:.4f}")

## Figures

In [ ]:
# bin_fig01 -- headline metrics
mets = [("Accuracy", "Accuracy"), ("MCC", "MCC"), ("Recall (malicious)", "Attack recall"), ("Specificity (benign recall)", "Benign recall"), ("F1 (malicious)", "Attack F1")]
fig, ax = plt.subplots(figsize=(8.5, 3.6)); w = .8 / len(present)
for j, b in enumerate(present):
    r = b1[b1.Dataset == lab(b)].iloc[0]
    ax.bar(np.arange(len(mets)) + j * w, [r[k] for k, _ in mets], w, label=lab(b), color=col(b))
    for i, (k, _) in enumerate(mets): ax.text(i + j * w, r[k] + .005, f"{r[k]:.3f}", ha="center", fontsize=6.5, rotation=90)
ax.set_xticks(np.arange(len(mets)) + w * (len(present) - 1) / 2, [l for _, l in mets]); ax.set_ylim(0, 1.12); ax.legend(loc="lower left", fontsize=7)
ax.set_title("Binary task: held-out test metrics of the optimized decision tree")
save_fig(fig, FIG / "bin_fig01_headline"); plt.show()

# bin_fig02 -- detection rate per attack type
have = [b for b in present if AT[b] is not None]
if have:
    fig, axes = plt.subplots(1, len(have), figsize=(5.6 * len(have), 4.2), squeeze=False)
    for ax, b in zip(axes[0], have):
        a = AT[b][AT[b].kind == "attack"].sort_values("n_test")
        ax.barh(a["attack_type"], a["rate"], color=["#c0392b" if r < .5 else "#e67e22" if r < .9 else "#27ae60" for r in a["rate"]])
        for i, (r_, n_) in enumerate(zip(a["rate"], a["n_test"])): ax.text(min(r_ + .01, .97), i, f"{r_:.2f} (n={n_:,})", va="center", fontsize=6.5)
        fpr = AT[b][AT[b].kind == "benign"]["rate"].mean()
        ax.set_xlim(0, 1.25); ax.set_xticks(np.arange(0, 1.01, .2)); ax.set_xlabel("detection rate"); ax.set_title(f"{lab(b)}\nbenign false-positive rate {fpr:.4f}", fontsize=9)
    fig.suptitle("Which attack types does the binary detector miss?  (red < 0.5, orange < 0.9, green >= 0.9)")
    save_fig(fig, FIG / "bin_fig02_attack_type_detection"); plt.show()

# bin_fig03 -- confusion matrices
fig, axes = plt.subplots(1, len(present), figsize=(4.4 * len(present), 3.8), squeeze=False)
for ax, b in zip(axes[0], present):
    cm = CM[b].values; nm = cm / cm.sum(axis=1, keepdims=True); ax.imshow(nm, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
    for i in range(2):
        for k in range(2): ax.text(k, i, f"{nm[i, k]:.3f}\n({cm[i, k]:,})", ha="center", va="center", fontsize=8, color="white" if nm[i, k] > .5 else "black")
    ax.set_xticks([0, 1], ["Benign", "Malicious"]); ax.set_yticks([0, 1], ["Benign", "Malicious"]); ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(lab(b), fontsize=9)
fig.suptitle("Row-normalised confusion matrices (binary task)"); save_fig(fig, FIG / "bin_fig03_confusion"); plt.show()

# bin_fig04 -- baselines
fig, axes = plt.subplots(2, len(present), figsize=(4.8 * len(present), 6.4), squeeze=False)
for j, b in enumerate(present):
    t = BL[b].dropna(subset=["mcc"]); t = t[~t.model.str.contains("Dummy")].sort_values("mcc")
    ax = axes[0, j]; ax.barh(t["model"], t["mcc"], color=[HIGHLIGHT if "Optimized" in m else NEUTRAL for m in t["model"]]); ax.set_xlabel("test MCC"); ax.set_title(lab(b), fontsize=9); ax.tick_params(axis="y", labelsize=6)
    ax = axes[1, j]; u = t.dropna(subset=["size_proxy_kb"])
    ax.scatter(u["size_proxy_kb"], u["mcc"], c=[HIGHLIGHT if "Optimized" in m else NEUTRAL for m in u["model"]], s=40)
    for _, r in u.iterrows():
        if "Optimized" in r["model"] or "FULLTRAIN" in r["model"]: ax.annotate(r["model"].replace("_best_k", "").replace("_features", ""), (r["size_proxy_kb"], r["mcc"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("size proxy (KB, log; single trees only)"); ax.set_ylabel("test MCC")
fig.suptitle("Binary task: optimized tree vs same-split baselines"); save_fig(fig, FIG / "bin_fig04_baselines"); plt.show()